# 18.11 综合实践三：编辑效应与基因组预测的多性状综合选择

《统计遗传与智能育种学》第十八章 · 电子资源 `18c_breeding_decision.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：18c_breeding_decision.ipynb。实践模拟240个候选材料的产量、蛋白和熟期GEBV，不在本章训练或验证GS模型。代码把编辑效应点估计及其标准误按正态近似计算单侧95%收益下限，再按产量等综合评分及蛋白、熟期、风险与收益下限约束选择，并比较仅依据编辑效应排序与纳入多性状育种目标和风险约束后的选择结果。

运行检查：重启内核并运行全部单元。本书实际运行中，240个候选有122个满足全部约束；只按edit_gain选择的20个材料有10个违反预设约束，而综合选择的20个违反数为0。两种方案仅重叠6个材料。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1803；240个模拟育种背景；3个模拟性状GEBV；编辑效应与标准误；脱靶风险；固定的蛋白、熟期、风险和编辑收益下界约束；无外部数据文件。

In [1]:
import sys
import numpy as np
import pandas as pd

SEED = 1803
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('numpy', np.__version__, 'pandas', pd.__version__)
print('SEED', SEED)

n = 240
cov = np.array([[1.0, -0.28, 0.18], [-0.28, 1.0, 0.05], [0.18, 0.05, 1.0]])
g = rng.multivariate_normal(np.zeros(3), cov, size=n)
cand = pd.DataFrame({
    'line': [f'L{i:03d}' for i in range(n)],
    'gebv_yield': g[:, 0],
    'gebv_protein': g[:, 1],
    'gebv_maturity_days': 112 + 3.2 * g[:, 2],
    'edit_gain': rng.normal(.32, .13, n) + .08 * g[:, 0],
    'edit_se': rng.uniform(.05, .16, n),
    'offtarget_risk': rng.beta(1.5, 4.5, n),
})
cand['edit_lcb'] = cand.edit_gain - 1.64 * cand.edit_se
for c in ['gebv_yield', 'gebv_protein', 'edit_lcb']:
    cand['z_' + c] = (cand[c] - cand[c].mean()) / cand[c].std(ddof=0)
cand['z_earliness'] = -(cand.gebv_maturity_days - cand.gebv_maturity_days.mean()) / cand.gebv_maturity_days.std(ddof=0)
cand['index'] = (0.45 * cand.z_gebv_yield + 0.25 * cand.z_gebv_protein
                 + 0.15 * cand.z_earliness + 0.15 * cand.z_edit_lcb)
cand['feasible'] = ((cand.gebv_protein >= -0.60)
                    & (cand.gebv_maturity_days <= 116.0)
                    & (cand.offtarget_risk <= .45)
                    & (cand.edit_lcb > 0))
k = 20
edit_only = cand.nlargest(k, 'edit_gain')
integrated = cand[cand.feasible].nlargest(k, 'index')

def summarize(name, frame):
    violations = int((~frame.feasible).sum())
    print(name, 'n', len(frame), 'violations', violations,
          'yield', round(frame.gebv_yield.mean(), 3),
          'protein', round(frame.gebv_protein.mean(), 3),
          'maturity_days', round(frame.gebv_maturity_days.mean(), 2),
          'edit_gain', round(frame.edit_gain.mean(), 3),
          'edit_lcb', round(frame.edit_lcb.mean(), 3))

print('feasible_candidates', int(cand.feasible.sum()), 'of', n)
summarize('EDIT_ONLY', edit_only)
summarize('INTEGRATED', integrated)
print('selected_overlap', len(set(edit_only.line) & set(integrated.line)), 'of', k)
print(integrated[['line', 'index', 'gebv_yield', 'gebv_protein',
                  'gebv_maturity_days', 'edit_gain', 'edit_lcb', 'offtarget_risk']]
      .head(8).round(3).to_string(index=False))


Python 3.12.13
numpy 2.3.5 pandas 2.2.3
SEED 1803
feasible_candidates 122 of 240
EDIT_ONLY n 20 violations 10 yield 0.972 protein -0.406 maturity_days 112.41 edit_gain 0.584 edit_lcb 0.412
INTEGRATED n 20 violations 0 yield 1.185 protein 0.506 maturity_days 111.09 edit_gain 0.447 edit_lcb 0.277
selected_overlap 6 of 20
line  index  gebv_yield  gebv_protein  gebv_maturity_days  edit_gain  edit_lcb  offtarget_risk
L217  1.757       1.970        -0.301             101.048      0.694     0.599           0.275
L020  1.391       2.769        -0.278             113.322      0.604     0.436           0.210
L227  1.163       2.038         0.031             115.815      0.670     0.571           0.156
L084  1.128       1.555         0.829             112.492      0.601     0.389           0.296
L056  0.901      -0.001         2.260             109.919      0.567     0.370           0.424
L004  0.847       0.442         1.034             106.083      0.353     0.239           0.075
L148  0.830   

结果解释：编辑收益最高不等于整体育种价值最高。综合方案的edit_gain点估计较低，产量与蛋白GEBV均值较高，并满足全部预设约束。这里的数值来自受控模拟，不代表任何真实育种材料的应用效果；真实应用还必须把GEBV外部验证误差和编辑×背景/环境不确定性带入决策。

可修改练习：把蛋白约束从−0.60提高到0；再降低产量权重、提高蛋白权重；最后把edit_se整体扩大50%，比较可行候选数量与最终选择名单。